# 46. mIoU, Pixel Accuracy, Confusion Matrix 평가

이 노트북은 semantic segmentation 평가 metric을 직접 계산합니다.

이번 노트북의 목표는 다음과 같습니다.

- confusion matrix를 pixel 단위로 만드는 방법을 이해합니다.
- pixel accuracy와 mean IoU의 차이를 확인합니다.
- `ignore_index`를 metric에서 제외하는 방법을 익힙니다.

In [ ]:
import numpy as np

num_classes = 3
target = np.array([
    [0, 0, 0, 1, 1],
    [0, 0, 1, 1, 1],
    [0, 2, 2, 2, 1],
    [0, 2, 2, 2, 2],
])

pred = np.array([
    [0, 0, 1, 1, 1],
    [0, 0, 1, 0, 1],
    [0, 2, 0, 2, 1],
    [0, 2, 2, 1, 2],
])

## 46-1. Pixel accuracy

In [ ]:
pixel_acc = (pred == target).mean()
print("pixel accuracy:", round(pixel_acc, 4))

## 46-2. Confusion matrix

In [ ]:
def confusion_matrix(pred, target, num_classes, ignore_index=None):
    pred = pred.reshape(-1)
    target = target.reshape(-1)
    if ignore_index is not None:
        keep = target != ignore_index
        pred = pred[keep]
        target = target[keep]

    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(target, pred):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t, p] += 1
    return cm


cm = confusion_matrix(pred, target, num_classes)
print(cm)

confusion matrix에서 row는 ground truth, column은 prediction으로 둡니다.

```text
cm[class, class] = true positive
row sum = 해당 class의 실제 픽셀 수
column sum = 해당 class로 예측한 픽셀 수
```

## 46-3. Class IoU와 mean IoU

In [ ]:
def iou_from_confusion_matrix(cm):
    tp = np.diag(cm)
    gt = cm.sum(axis=1)
    pred_count = cm.sum(axis=0)
    union = gt + pred_count - tp
    iou = np.divide(tp, union, out=np.full_like(tp, np.nan, dtype=float), where=union != 0)
    return iou


ious = iou_from_confusion_matrix(cm)
for cls, iou in enumerate(ious):
    print(f"class {cls} IoU:", round(float(iou), 4))
print("mean IoU:", round(float(np.nanmean(ious)), 4))

## 46-4. ignore_index 처리

In [ ]:
target_with_ignore = target.copy()
target_with_ignore[0, 2] = 255

cm_ignore = confusion_matrix(pred, target_with_ignore, num_classes, ignore_index=255)
ious_ignore = iou_from_confusion_matrix(cm_ignore)

print(cm_ignore)
print("mean IoU with ignore_index:", round(float(np.nanmean(ious_ignore)), 4))

## 정리

- pixel accuracy는 전체 픽셀 중 맞춘 비율입니다.
- mIoU는 class별 영역 겹침을 평균하므로 class imbalance에 더 민감합니다.
- loss와 metric에서 `ignore_index` 처리 기준이 같아야 합니다.
- 다음 노트북 `47_Augmentation과_입출력_해상도_실험.ipynb`에서는 augmentation과 resolution 실험 기준을 다룹니다.